# Unit 2, Lecture 1: Workflow versus agency

One problem, built three ways. The point is not that the agent is best. The point
is that the simplest shape that works is usually the right one to ship, and you
can prove that with cold numbers rather than opinion.

The problem: **a support ticket comes in, route it to one of five queues**:
billing, technical, account, sales, abuse. That is the whole task.

## Setup

In [ ]:
from cse476.lanes import get_client, MODEL, describe
from cse476.triage import (
    SAMPLE_TICKETS, QUEUES, compare,
    triage_workflow, triage_router, triage_agent,
)

print(describe())
client = get_client()
print("queues:", QUEUES)
print()
for name, text in SAMPLE_TICKETS.items():
    print(f"  {name:10} {text}")

## Build one: the workflow

No model at all. Fixed keyword rules, same order every time. This is the baseline
everything else is measured against.

In [ ]:
for name, text in SAMPLE_TICKETS.items():
    t = triage_workflow(text)
    print(f"{name:10} -> {t.queue:10} ({t.model_calls} calls)  {t.reason}")

It nailed the easy ticket for **zero cost**. Now look at the `ambiguous` one:
"Nothing works and I want my money back."

Is that technical, or billing? The keyword rules grab whichever matches first,
with no understanding of intent. **That is the workflow hitting its ceiling**, and
it is the first honest reason to bring in a model.

## Build two: the router

One model call classifies the ticket into a queue. Then ordinary code takes over.
This is the shape most real "AI routing" systems should be.

In [ ]:
for name, text in SAMPLE_TICKETS.items():
    t = triage_router(client, MODEL, text)
    print(f"{name:10} -> {t.queue:10} ({t.model_calls} call)   {t.reason}")

The router understands the ambiguous ticket, because it reads intent rather
than keywords. But it still makes exactly **one** decision, from a fixed menu, and
the cost is bounded and known: one call, every time.

### The line that saves the router

```python
queue = choice if choice in QUEUES else "technical"
```

The model was asked for one word from a fixed list. A language model can always
return something off-menu. So we validate against the whitelist and default
anything invalid, exactly the same rule as validating a tool name in Unit 1.
Anything a model produces is untrusted input.

## Build three: the agent

Now the model decides the whole path, using tools to inspect the queues and their
policies, until it is sure. This is the only one of the three that is actually an
agent. It is your `tiny_agent` from Unit 1, pointed at triage.

In [ ]:
for name, text in SAMPLE_TICKETS.items():
    t = triage_agent(client, MODEL, text)
    print(f"{name:10} -> {t.queue:10} ({t.model_calls} calls)  {t.reason[:50]}")

## The punchline

Put all three on the same ticket, side by side.

In [ ]:
ticket = SAMPLE_TICKETS["easy"]   # "I was charged twice..."
print(f"ticket: {ticket}\n")

results = {
    "workflow": triage_workflow(ticket),
    "router":   triage_router(client, MODEL, ticket),
    "agent":    triage_agent(client, MODEL, ticket),
}
print(compare(results))

Same queue, every time. But roughly **0, 1, and 3 model calls**.

For a straightforward ticket, the agent paid three times what the router paid for
an identical outcome. The exact numbers vary run to run because the model chooses,
but the direction is stable and it is the whole lesson.

Multiply that gap by a million tickets a month and the choice of shape stops being
a style question and becomes a line item on a budget.

## The test you can apply to anything

Three questions, in order. Stop at the first yes.

1. **Can I write the steps in advance?** → build a workflow.
2. **Is it one decision, then a fixed path?** → build a router.
3. **Do the steps genuinely depend on what I find along the way?** → only now, an agent.

Triage is a question-two problem. That is why the router won.

## Your turn

**1. Add a sixth queue.** Pick one, for example `partnerships`. Update all three
builds to handle it. Notice which was easiest to change and which was hardest, and
write one sentence on why.

**2. Find the crossover.** Write three tickets: one the workflow handles fine, one
that needs at least the router, and one you believe genuinely needs the agent. Run
all three builds on each. Then defend your third ticket honestly, and if you
conclude it does not really need an agent after all, say so. That is the lesson
landing, not a failure.

**3. Measure the real cost.** Run each build on all four sample tickets and total
the model calls. That total, times a realistic monthly ticket volume, is the
number a real engineering decision would actually turn on.

In [ ]:
# your work here
